# Bozorgmehr-puncfixer fine-tuning

**Base model:** [`HooshvareLab/albert-fa-zwnj-base-v2`](https://huggingface.co/HooshvareLab/albert-fa-zwnj-base-v2) &nbsp;|&nbsp; **Dataset:** [`MohammadJRanjbar/persian-punctuation-restoration`](https://huggingface.co/datasets/MohammadJRanjbar/persian-punctuation-restoration)

## 1. Project overview and task definition

**Task.** Token-level sequence labelling. Input: a Persian sentence *without* punctuation. Output: for every word, the punctuation mark that follows it. The model never generates text, so the words of the output are, by construction, the words of the input (this is also verified by tests and at inference time).

| Label | Mark | Label | Mark |
|---|---|---|---|
| `NONE` | (nothing) | `EXCLAMATION` | `!` |
| `COMMA` | `،` | `SEMICOLON` | `؛` |
| `PERIOD` | `.` | `COLON` | `:` |
| `QUESTION` | `؟` | | |

**Important data fact.** The dataset contains only *punctuated reference sentences*. Inputs and targets are both derived from the same sentence by `parse_punctuated` (section 5): punctuation is removed to create the input, and each removed mark becomes the label of the word before it.

**Label-design policy (every exclusion is counted and reported, nothing is silently dropped)**
* `,`→`،` and `?`→`؟` (not when between two digits, e.g. `1,000`). A mark between two digits (`3.5`, `10:30`) belongs to the number.
* Unsupported symbols (quotes `« »`, brackets, hyphens...) stay *inside* the lexical token and are never predicted.
* A sentence is **excluded** (with its reason counted) if one label per word cannot represent it losslessly: consecutive marks (`؟!`, `...`), leading punctuation, or tokens made only of symbols (e.g. `،»`). The exclusion rate is measured in section 7 and on every evaluation.

**Alignment.** Each word is tokenised on its own; its label goes on its **last sub-word**; `[CLS]`, `[SEP]`, padding and non-final sub-words get `-100` and are ignored by the loss. Truncation happens at word boundaries, so labels always stay aligned. Section 6 tests this and compares it with the tokenizer's own `word_ids()` alignment.

**Training workflow.** (preflight) → unit tests → smoke test → pilot run (throughput/memory) → **resumable** streaming fine-tuning with atomic, verified checkpoints on Google Drive → validation-based checkpoint selection → one final test evaluation → inference + verified save.

**How to run:** Runtime → *Change runtime type* → **T4 GPU**. Run cells top to bottom. You need a Hugging Face account that has accepted the dataset conditions (section 4). All outputs go to `/content/drive/MyDrive/PersianPunc_ALBERT/`.

## 2. Environment and GPU verification

In [ ]:
# Colab already ships a CUDA build of PyTorch - we deliberately do NOT reinstall it.
# If pip reports that it changed an already-imported package, use Runtime -> Restart session and run again.
!pip install -q -U "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" "huggingface_hub>=0.24" \
    evaluate scikit-learn sentencepiece protobuf safetensors matplotlib pandas tqdm

In [ ]:
import os, sys, json, time, math, random, shutil, hashlib, copy, subprocess, warnings, itertools
from collections import Counter
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import transformers, datasets, accelerate, huggingface_hub, safetensors, sklearn
from safetensors.torch import load_file as st_load_file
from transformers import (AlbertForTokenClassification, AutoConfig, AutoTokenizer,
                          get_linear_schedule_with_warmup)
from datasets import load_dataset, load_dataset_builder

warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*before.*optimizer.step.*")

# ---------------------------------------------------------------- CONFIGURATION (single place) ----
@dataclass
class Cfg:
    # resources
    model_name: str = "HooshvareLab/albert-fa-zwnj-base-v2"
    dataset_name: str = "MohammadJRanjbar/persian-punctuation-restoration"
    drive_root: str = "/content/drive/MyDrive/bozorgmehr-puncfixer"
    # data
    max_length: int = 256            # sub-word positions incl. [CLS]/[SEP]; checked against the length distribution in sec. 7
    shuffle_buffer: int = 20_000     # streaming shuffle buffer (bounded RAM)
    buffer_steps: int = 100          # optimizer steps worth of examples preprocessed at once (bounded RAM); checkpoints/evals happen at these boundaries
    integrity_sample: int = 100_000  # rows inspected in the data-integrity report
    # optimisation (initial values from the task statement; treat as starting points)
    train_bs: int = 16
    eval_bs: int = 32
    grad_accum: int = 2
    lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    max_grad_norm: float = 1.0
    seed: int = 42
    fp16: bool = True
    # schedule / bookkeeping
    max_train_steps: int = 20_000    # OPTIMIZER steps in this campaign (= LR schedule length). 20k steps x 32 seq = 640k sentences (~3.8% of train)
    logging_steps: int = 100
    eval_steps: int = 1000
    save_steps: int = 1000
    early_stopping_patience: int = 3 # in eval rounds; 0 disables. Counter is persisted, so resuming is always possible
    val_limit_during_training: int = 5000   # validation sentences used for model selection during training (full validation at the end)
    eval_chunk: int = 4096           # sentences preprocessed at once during evaluation
    # session control
    session_time_budget_min: int = 0 # >0: stop gracefully (after saving a checkpoint) once this many minutes have elapsed
    resume: str = "auto"             # "auto" | "none" | "/path/to/checkpoint/step-000001000"
    resume_mode: str = "skip"        # "skip": fast-forward the seeded stream by examples_read (exact);  "state_dict": datasets' own stream state (faster for deep resumes, shuffle-buffer content is not restored)
    allow_fresh_start: bool = False  # must be True to start from step 0 when valid checkpoints already exist
    allow_schedule_change: bool = False  # True: allow raising max_train_steps when resuming (LR schedule is re-derived; LR may jump)
    delete_old_checkpoints: bool = False # destructive, opt-in
    keep_last: int = 3
    max_oom_skips: int = 5
    # pilot / evaluation
    run_pilot: bool = True
    force_pilot: bool = False
    pilot_steps: int = 300
    pilot_eval_limit: int = 2000
    run_training: bool = True
    test_limit: Optional[int] = None  # None = full published test split (327,939). An int evaluates the FIRST n sentences only and is reported as a subset
    force_integrity: bool = False
    allow_cpu: bool = False           # CPU is only for debugging; training on CPU is impractical

cfg = Cfg()
EXPECTED_SPLITS = {"train": 16_952_014, "validation": 17_432, "test": 327_939}

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(cfg.seed)

# ---------------------------------------------------------------- GPU verification ----------------
HAVE_CUDA = torch.cuda.is_available()
if not HAVE_CUDA:
    msg = "No CUDA GPU. In Colab: Runtime -> Change runtime type -> T4 GPU, then restart and run again."
    if not cfg.allow_cpu:
        raise RuntimeError(msg)
    print("WARNING:", msg)
DEVICE = torch.device("cuda" if HAVE_CUDA else "cpu")
GPU_INFO = {}
if HAVE_CUDA:
    p = torch.cuda.get_device_properties(0)
    GPU_INFO = {"name": p.name, "total_gb": round(p.total_memory / 2**30, 2), "capability": f"{p.major}.{p.minor}"}
    if "T4" not in p.name:
        print(f"NOTE: expected a Tesla T4, found {p.name}. Settings are still valid; throughput will differ.")
    x = torch.randn(256, 256, device="cuda", dtype=torch.float16)
    assert torch.isfinite(x @ x).all(), "fp16 matmul sanity check failed"
USE_FP16 = bool(cfg.fp16 and HAVE_CUDA and torch.cuda.get_device_capability(0) >= (7, 0))

VERSIONS = {"python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.version.cuda,
            "transformers": transformers.__version__, "datasets": datasets.__version__,
            "accelerate": accelerate.__version__, "huggingface_hub": huggingface_hub.__version__,
            "safetensors": safetensors.__version__, "scikit-learn": sklearn.__version__,
            "numpy": np.__version__, "pandas": pd.__version__}
print(pd.Series({**VERSIONS, **{"gpu_" + k: v for k, v in GPU_INFO.items()}, "fp16_autocast": USE_FP16}).to_string())

## 3. Google Drive setup
Everything that must survive a Colab disconnect (checkpoints, logs, final model) is written under `cfg.drive_root`. Nothing important lives only in `/content`.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # follow the authorisation prompt
else:
    print("Not running in Colab: using the local path from cfg.drive_root instead of Google Drive.")

ROOT = Path(cfg.drive_root)
DIRS = {k: ROOT / v for k, v in dict(model="model", checkpoints="checkpoints", logs="logs", evaluation="evaluation",
                                    cache="cache", config="config", examples="examples").items()}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
PKG = Path("/content/punc_pkg"); PKG.mkdir(parents=True, exist_ok=True)   # code written in section 5, copied to Drive in section 14
sys.path.insert(0, str(PKG))
probe = ROOT / ".write_probe"; probe.write_text("ok"); assert probe.read_text() == "ok"; probe.unlink()
print("Drive outputs ->", ROOT)
for k, v in DIRS.items():
    print(f"  {k:12s}{v}")

## 4. Dataset and model inspection
**Access.** The dataset repository is *publicly listed but gated*: you must (1) be logged in to Hugging Face and (2) open the [dataset page](https://huggingface.co/datasets/MohammadJRanjbar/persian-punctuation-restoration) once and accept the conditions. Then provide a token with *read* access in one of two ways: add a Colab secret named `HF_TOKEN` (key icon in the left sidebar, enable notebook access) **or** paste it into the login widget that appears below. The token is never written to the notebook, config or Drive.

In [ ]:
from huggingface_hub import login, get_token
def ensure_hf_login() -> None:
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = os.environ.get("HF_TOKEN")
    if token:
        login(token=token, add_to_git_credential=False); print("Logged in with HF_TOKEN secret.")
    elif get_token() is not None:
        print("Using an existing Hugging Face login.")
    else:
        from huggingface_hub import notebook_login
        print("No token found: paste a READ token below, then run the next cell.")
        notebook_login()
ensure_hf_login()

In [ ]:
ACCESS_HELP = ("Cannot access the dataset. Do this once: (1) log in to huggingface.co, (2) open "
               f"https://huggingface.co/datasets/{cfg.dataset_name} and accept the access conditions, "
               "(3) create a READ token at https://huggingface.co/settings/tokens and store it as Colab secret HF_TOKEN "
               "(or paste it in the login widget above), (4) re-run this cell.")
try:
    builder = load_dataset_builder(cfg.dataset_name)
except Exception as e:
    print(ACCESS_HELP); raise

print("Declared features:", builder.info.features)
declared_splits = {}
try:
    declared_splits = {k: v.num_examples for k, v in (builder.info.splits or {}).items()}
except Exception:
    pass
print("Declared split sizes:", declared_splits or "not declared in repo metadata (will be checked after loading)")

# ---- tokenizer & config (inspected BEFORE building the task head) -------------------------------
try:
    tok = AutoTokenizer.from_pretrained(cfg.model_name)
except Exception as e:
    print("Fast tokenizer conversion failed (", repr(e)[:120], ") -> falling back to the slow SentencePiece tokenizer")
    tok = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=False)
mcfg = AutoConfig.from_pretrained(cfg.model_name)
print("\nTokenizer:", type(tok).__name__, "| fast:", tok.is_fast, "| len:", len(tok), "| model_max_length:", tok.model_max_length)
print("Special tokens:", {k: (v, tok.convert_tokens_to_ids(v)) for k, v in tok.special_tokens_map.items()})
print("Tokenizer init kwargs of interest:", {k: tok.init_kwargs.get(k) for k in ("do_lower_case", "keep_accents", "remove_space")})
print("Model config:", {k: getattr(mcfg, k, None) for k in ("model_type", "architectures", "vocab_size", "embedding_size", "hidden_size",
      "num_hidden_layers", "num_hidden_groups", "num_attention_heads", "intermediate_size", "max_position_embeddings", "classifier_dropout_prob")})
assert mcfg.model_type == "albert", "unexpected architecture"
assert None not in (tok.cls_token_id, tok.sep_token_id, tok.pad_token_id), "tokenizer lacks [CLS]/[SEP]/[PAD]"
assert mcfg.max_position_embeddings >= cfg.max_length, f"cfg.max_length exceeds the model's {mcfg.max_position_embeddings} positions"
assert len(tok) <= mcfg.vocab_size, "tokenizer larger than the embedding matrix"
for w in ["می\u200cروم", "کتاب\u200cها", "سلام", "ي", "ك"]:
    print(f"  {w!r:14} -> {tok.tokenize(w)}")

In [ ]:
# ---- dataset: schema, sizes and samples ------------------------------------------------------------
HF_CACHE = str(DIRS["cache"] / "hf_datasets")
train_stream_probe = load_dataset(cfg.dataset_name, split="train", streaming=True)
print("Train features:", train_stream_probe.features, "| columns:", train_stream_probe.column_names)
val_ds = load_dataset(cfg.dataset_name, split="validation", cache_dir=HF_CACHE)
print("Validation:", val_ds)
for col in ("sentence", "source_dataset", "punctuation_count", "sentence_length"):
    assert col in val_ds.column_names, f"expected column {col!r} is missing"
assert val_ds.features["sentence"].dtype == "string"
if len(val_ds) != EXPECTED_SPLITS["validation"]:
    print(f"WARNING: validation has {len(val_ds)} rows, the dataset card says {EXPECTED_SPLITS['validation']}")
val_sentences: List[str] = list(val_ds["sentence"])

_TEST_CACHE: Dict[str, List[str]] = {}
def get_test_sentences() -> List[str]:
    """Loads the published test split (downloads only the test files). Never used for gradient updates."""
    if "test" not in _TEST_CACHE:
        ds = load_dataset(cfg.dataset_name, split="test", cache_dir=HF_CACHE)
        if len(ds) != EXPECTED_SPLITS["test"]:
            print(f"WARNING: test has {len(ds)} rows, the dataset card says {EXPECTED_SPLITS['test']}")
        _TEST_CACHE["test"] = list(ds["sentence"])
    return _TEST_CACHE["test"]

print("\nSample records (trimmed to 160 chars):")
for r in itertools.islice(train_stream_probe, 4):
    print({k: (v[:160] if isinstance(v, str) else v) for k, v in r.items()})
print("\nValidation sources:", dict(Counter(val_ds["source_dataset"]).most_common(8)))

In [ ]:
# ---- preflight report ---------------------------------------------------------------------------------
PREFLIGHT = {
    "versions": VERSIONS, "gpu": GPU_INFO, "fp16_autocast": USE_FP16,
    "model": {"name": cfg.model_name, "tokenizer": type(tok).__name__, "fast": tok.is_fast, "len_vocab": len(tok),
              "max_position_embeddings": mcfg.max_position_embeddings},
    "dataset": {"name": cfg.dataset_name, "columns": val_ds.column_names, "declared_splits": declared_splits,
                "expected_splits_from_card": EXPECTED_SPLITS, "validation_rows_loaded": len(val_ds)},
    "assumptions": ["Train split size is taken from the dataset card (streaming gives no count).",
                    "Token 'sanitised' samples are trimmed sentences; no credentials are stored anywhere."],
}
(DIRS["config"] / "preflight.json").write_text(json.dumps(PREFLIGHT, ensure_ascii=False, indent=2))
print(json.dumps({k: PREFLIGHT[k] for k in ("versions", "gpu", "fp16_autocast")}, indent=1))

## 5. Punctuation extraction and label construction
The text logic lives in two small, dependency-free files that are written to disk here, imported by this notebook, and later shipped with the model:
* `punc_core.py` - label inventory, `parse_punctuated` (reference sentence → words + labels), `reconstruct`, `encode_words` (label ↔ sub-word alignment), long-input windowing;
* `punc_inference.py` - the standalone `PunctuationRestorer` (section 13).

In [ ]:
%%writefile /content/punc_pkg/punc_core.py
"""punc_core.py - dependency-free text logic for Persian punctuation restoration.

Single source of truth shared by training, evaluation and inference.

Policy summary (all measured/reported by the notebook, nothing silently dropped):
  * Label inventory: NONE, COMMA(،) PERIOD(.) QUESTION(؟) EXCLAMATION(!) SEMICOLON(؛) COLON(:)
  * ',' -> '،' and '?' -> '؟' (except between two digits, e.g. "1,000" stays as is).
  * A punctuation mark between two digits (3.5, 10:30) is part of the lexical token.
  * Any other punctuation mark belongs to the label of the word it follows
    (with or without whitespace in between). Space-before-punctuation is normalised.
  * Unsupported symbols (quotes, brackets, hyphens, ...) stay INSIDE the lexical token
    and are never predicted.
  * Examples that cannot be represented losslessly by ONE label per word are EXCLUDED
    from training/evaluation (and counted by reason): consecutive marks ("?!", "..."),
    leading punctuation, and tokens made only of symbols (e.g. a closing quote after
    punctuation: `،»`).
"""
from __future__ import annotations

import re
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Sequence, Tuple

LABELS: List[str] = ["NONE", "COMMA", "PERIOD", "QUESTION", "EXCLAMATION", "SEMICOLON", "COLON"]
LABEL2ID: Dict[str, int] = {l: i for i, l in enumerate(LABELS)}
ID2LABEL: Dict[int, str] = {i: l for l, i in LABEL2ID.items()}
LABEL2PUNCT: Dict[str, str] = {
    "NONE": "", "COMMA": "،", "PERIOD": ".", "QUESTION": "؟",
    "EXCLAMATION": "!", "SEMICOLON": "؛", "COLON": ":",
}
ID2PUNCT: List[str] = [LABEL2PUNCT[l] for l in LABELS]
# every character that is treated as a punctuation event (incl. ASCII variants)
PUNCT_TO_LABEL: Dict[str, int] = {
    "،": 1, ",": 1, ".": 2, "؟": 3, "?": 3, "!": 4, "؛": 5, ":": 6,
}
IGNORE_INDEX = -100
_WS = re.compile(r"\s+")


@dataclass
class ParseResult:
    words: List[str]
    labels: List[int]
    reason: Optional[str] = None      # None => valid example
    symbols: Tuple[str, ...] = ()     # unsupported symbols kept inside tokens


def _numeric_context(t: str, i: int) -> bool:
    """True if t[i] sits between two digits (3.5, 10:30, 1,000)."""
    return 0 < i < len(t) - 1 and t[i - 1].isdigit() and t[i + 1].isdigit()


def normalize_whitespace(text: str) -> str:
    return _WS.sub(" ", text).strip()


def normalize_text(text: str) -> str:
    """Reference normalisation: whitespace collapse + ASCII ','/'?' -> Persian (not inside numbers)."""
    t = normalize_whitespace(text)
    out = []
    for i, c in enumerate(t):
        if c in ",?" and not _numeric_context(t, i):
            c = "،" if c == "," else "؟"
        out.append(c)
    return "".join(out)


def _scan(text: str, strict: bool) -> ParseResult:
    if not isinstance(text, str):
        return ParseResult([], [], "not_a_string")
    t = normalize_whitespace(text)
    if not t:
        return ParseResult([], [], "empty")
    words: List[str] = []
    labels: List[int] = []
    buf: List[str] = []
    i, n = 0, len(t)
    while i < n:
        c = t[i]
        if c == " ":
            if buf:
                words.append("".join(buf)); labels.append(0); buf = []
            i += 1
            continue
        if c in PUNCT_TO_LABEL:
            if _numeric_context(t, i):
                buf.append(c); i += 1
                continue
            j = i
            while j < n and t[j] in PUNCT_TO_LABEL:
                j += 1
            if j - i > 1 and strict:
                return ParseResult([], [], "multi_punct")
            lab = PUNCT_TO_LABEL[c]
            if buf:
                words.append("".join(buf)); labels.append(lab); buf = []
            elif words and labels[-1] == 0:
                labels[-1] = lab                      # "word ،"  (space before punctuation)
            elif words:
                if strict:
                    return ParseResult([], [], "multi_punct")
            else:
                if strict:
                    return ParseResult([], [], "leading_punct")
            i = j
            continue
        buf.append(c)
        i += 1
    if buf:
        words.append("".join(buf)); labels.append(0)
    if not words:
        return ParseResult([], [], "empty_after_parse")
    if strict and any(not any(ch.isalnum() for ch in w) for w in words):
        return ParseResult([], [], "symbol_only_token")
    syms = tuple(sorted({ch for w in words for ch in w if not ch.isalnum() and ch != "\u200c"
                         and not (ch in PUNCT_TO_LABEL)}))
    return ParseResult(words, labels, None, syms)


def parse_punctuated(text: str) -> ParseResult:
    """Strict parse of a punctuated reference sentence -> (words, labels)."""
    return _scan(text, strict=True)


def split_lexical(text: str) -> List[str]:
    """Lenient parse: lexical tokens only (all supported punctuation removed)."""
    return _scan(text, strict=False).words


def reconstruct(words: Sequence[str], label_ids: Sequence[int]) -> str:
    assert len(words) == len(label_ids), "words/labels length mismatch"
    return " ".join(w + ID2PUNCT[int(l)] for w, l in zip(words, label_ids))


def spacing_insensitive(s: str) -> str:
    return re.sub(r"\s+", "", s)


def encode_words(
    words: Sequence[str], labels: Sequence[int],
    tokenize_words: Callable[[Sequence[str]], List[List[int]]],
    cls_id: int, sep_id: int, max_length: int,
) -> Tuple[Optional[dict], Optional[str]]:
    """Build [CLS] pieces [SEP] with each word's label on its LAST sub-word piece.

    Labels are derived from the original punctuated sentence (never from the stripped input),
    and every other position (special tokens, non-final pieces) gets -100. Truncation happens at
    WORD boundaries so labels always stay aligned with the retained input.
    """
    pieces = tokenize_words(words)
    budget = max_length - 2
    ids, lab, total, kept, truncated = [cls_id], [IGNORE_INDEX], 0, 0, False
    for w_i, p in enumerate(pieces):
        if not p:
            return None, "untokenizable_word"
        if total + len(p) > budget:
            truncated = True
            break
        ids.extend(p)
        lab.extend([IGNORE_INDEX] * (len(p) - 1) + [int(labels[w_i])])
        total += len(p); kept += 1
    if kept == 0:
        return None, "no_words_fit"
    ids.append(sep_id); lab.append(IGNORE_INDEX)
    return {"input_ids": ids, "labels": lab, "n_words": kept, "truncated": truncated}, None


def plan_windows(piece_counts: Sequence[int], budget: int, overlap_words: int = 16) -> List[Tuple[int, int]]:
    """Greedy windows over words so no input is silently truncated; consecutive windows overlap."""
    n, windows, start = len(piece_counts), [], 0
    while start < n:
        end, tot = start, 0
        while end < n and tot + piece_counts[end] <= budget:
            tot += piece_counts[end]; end += 1
        if end == start:           # a single word longer than the budget: its pieces get clipped
            end = start + 1
        windows.append((start, end))
        if end >= n:
            break
        start = max(end - overlap_words, start + 1)
    return windows


def best_window_for_each_word(n: int, windows: Sequence[Tuple[int, int]]) -> List[int]:
    """For each word choose the window in which it is farthest from a (non-document) edge."""
    big = 10 ** 9
    best, best_m = [0] * n, [-1] * n
    for w_idx, (s, e) in enumerate(windows):
        for i in range(s, e):
            m = min(i - s if s > 0 else big, e - 1 - i if e < n else big)
            if m > best_m[i]:
                best_m[i], best[i] = m, w_idx
    return best


In [ ]:
%%writefile /content/punc_pkg/punc_inference.py
"""punc_inference.py - standalone Persian punctuation restoration (needs only torch, transformers, punc_core).

    from punc_inference import PunctuationRestorer
    r = PunctuationRestorer("/path/to/model_dir")
    r.restore("سلام حال شما چطور است")           # -> "سلام، حال شما چطور است؟"
    r.restore(["...", "..."])                      # batch (order preserved)

Behaviour:
  * Input is reduced to lexical tokens with the same parser used in training: whitespace is collapsed
    and any punctuation already present in the input is DISCARDED and re-predicted
    (marks between digits, e.g. 3.5 or 10:30, are kept as part of the token).
  * Output words are exactly the input words (spelling, ZWNJ, order). This is verified on every call
    (verify=True) and a RuntimeError is raised if it ever fails.
  * Long inputs are never truncated: words are split into overlapping windows that fit max_length sub-words;
    each word takes its prediction from the window in which it is farthest from a window edge. There is no
    forced punctuation at segment boundaries; the model decides at every word using real context on both sides.
    A single word longer than the window has its sub-word pieces clipped for context only (the word itself is kept).
  * Words that the tokenizer maps to zero pieces receive NONE.
"""
from __future__ import annotations

from typing import Dict, List, Optional, Sequence, Union

import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer

from punc_core import (best_window_for_each_word, plan_windows, reconstruct, split_lexical)


class PunctuationRestorer:
    def __init__(self, model_dir: str, device: Optional[str] = None, max_length: int = 256,
                 overlap_words: int = 16, batch_size: int = 32, fp16: Optional[bool] = None):
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.tok = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForTokenClassification.from_pretrained(model_dir).to(self.device).eval()
        self.max_length, self.overlap, self.batch_size = max_length, overlap_words, batch_size
        self.fp16 = (self.device.type == "cuda") if fp16 is None else fp16
        self.cache: Dict[str, List[int]] = {}
        self.cls_id, self.sep_id, self.pad_id = self.tok.cls_token_id, self.tok.sep_token_id, self.tok.pad_token_id

    # ---- tokenisation -------------------------------------------------------------------
    def _pieces(self, words: Sequence[str]) -> List[List[int]]:
        if len(self.cache) > 1_000_000:
            self.cache.clear()
        miss = list({w for w in words if w not in self.cache})
        if miss:
            out = self.tok(miss, add_special_tokens=False, return_attention_mask=False,
                           return_token_type_ids=False)["input_ids"]
            self.cache.update(zip(miss, out))
        return [self.cache[w] for w in words]

    # ---- prediction ---------------------------------------------------------------------
    @torch.no_grad()
    def predict_labels(self, words_per_text: List[List[str]]) -> List[List[int]]:
        budget = self.max_length - 2
        jobs = []                                   # (text_idx, win_idx, start, end, ids, last_piece_positions)
        result: List[List[int]] = [[0] * len(w) for w in words_per_text]
        plans = []
        for t_idx, words in enumerate(words_per_text):
            pieces = self._pieces(words) if words else []
            counts = [max(len(p), 1) for p in pieces]
            wins = plan_windows(counts, budget, self.overlap)
            owner = best_window_for_each_word(len(words), wins)
            plans.append((wins, owner))
            for w_idx, (s, e) in enumerate(wins):
                ids, last = [self.cls_id], {}
                for i in range(s, e):
                    p = pieces[i][: max(budget - (len(ids) - 1), 0)]
                    if not p:
                        continue                    # zero-piece word -> stays NONE
                    ids.extend(p); last[i] = len(ids) - 1
                ids.append(self.sep_id)
                jobs.append((t_idx, w_idx, ids, last))
        order = sorted(range(len(jobs)), key=lambda j: len(jobs[j][2]))
        preds: Dict[int, torch.Tensor] = {}
        for b in range(0, len(order), self.batch_size):
            idx = order[b:b + self.batch_size]
            L = max(len(jobs[j][2]) for j in idx)
            ids = torch.full((len(idx), L), self.pad_id, dtype=torch.long)
            att = torch.zeros((len(idx), L), dtype=torch.long)
            for r, j in enumerate(idx):
                x = jobs[j][2]; ids[r, :len(x)] = torch.tensor(x); att[r, :len(x)] = 1
            with torch.autocast(device_type=self.device.type, dtype=torch.float16, enabled=self.fp16):
                logits = self.model(input_ids=ids.to(self.device), attention_mask=att.to(self.device)).logits
            p = logits.argmax(-1).cpu()
            for r, j in enumerate(idx):
                preds[j] = p[r]
        job_of = {(j[0], j[1]): k for k, j in enumerate(jobs)}
        for t_idx, (wins, owner) in enumerate(plans):
            for i, w_idx in enumerate(owner):
                k = job_of[(t_idx, w_idx)]
                pos = jobs[k][3].get(i)
                if pos is not None:
                    result[t_idx][i] = int(preds[k][pos])
        return result

    # ---- public API ---------------------------------------------------------------------
    def restore(self, text: Union[str, List[str]], verify: bool = True) -> Union[str, List[str]]:
        single = isinstance(text, str)
        texts = [text] if single else list(text)
        words_per_text = [split_lexical(t) if isinstance(t, str) else [] for t in texts]
        labels = self.predict_labels(words_per_text)
        outs = []
        for t, words, lab in zip(texts, words_per_text, labels):
            out = reconstruct(words, lab) if words else ""
            if verify and split_lexical(out) != words:
                raise RuntimeError(f"Lexical content changed during restoration for input: {t!r}")
            outs.append(out)
        return outs[0] if single else outs


In [ ]:
import importlib
import punc_core, punc_inference
importlib.reload(punc_core); importlib.reload(punc_inference)
from punc_core import (LABELS, LABEL2ID, ID2LABEL, LABEL2PUNCT, ID2PUNCT, IGNORE_INDEX, parse_punctuated, split_lexical,
                       reconstruct, normalize_text, spacing_insensitive, encode_words, plan_windows, best_window_for_each_word)
from punc_inference import PunctuationRestorer
C = len(LABELS)

demo = ["سلام، حال شما چطور است؟", "او گفت:«من می\u200cآیم».", "ساعت 10:30 است؛ برو!", "صبر کن...", "او گفت: «بیا.»"]
for s in demo:
    r = parse_punctuated(s)
    print(f"{s!r}\n   -> reason={r.reason} words={r.words} labels={[ID2LABEL[l] for l in r.labels]}")

## 6. Preprocessing unit tests
Tests run on pure text first (parsing, normalisation, reconstruction, lexical preservation, alignment with a mock tokenizer, truncation, windows), then on the **real tokenizer** (label position of every word, comparison with `word_ids()`).

In [ ]:
# ---- pure-text tests (identical to the tests executed during development) ----

def P(s): return parse_punctuated(s)

def check(s, words, labs, exact=True):
    r = P(s); assert r.reason is None, (s, r.reason)
    assert r.words == words, (r.words, words)
    assert r.labels == [LABEL2ID[l] for l in labs], (r.labels, labs)
    rec = reconstruct(r.words, r.labels)
    if exact: assert rec == normalize_text(s), (rec, normalize_text(s))
    else: assert spacing_insensitive(rec) == spacing_insensitive(normalize_text(s))
    assert split_lexical(rec) == r.words            # lexical preservation after reconstruction
    assert split_lexical(s) == r.words

N="NONE"
check("سلام، حال شما چطور است؟", ["سلام","حال","شما","چطور","است"], ["COMMA",N,N,N,"QUESTION"])
check("سیب, پرتقال, و موز.", ["سیب","پرتقال","و","موز"], ["COMMA","COMMA",N,"PERIOD"])      # ASCII comma
check("آیا می‌آیی?", ["آیا","می‌آیی"], [N,"QUESTION"])                                         # ZWNJ + ASCII ?
check("او گفت:«من می‌آیم».", ["او","گفت","«من","می‌آیم»"], [N,"COLON",N,"PERIOD"], exact=False)
check("قیمت 3.5 دلار است.", ["قیمت","3.5","دلار","است"], [N,N,N,"PERIOD"])
check("ساعت 10:30 است؛ برو!", ["ساعت","10:30","است","برو"], [N,N,"SEMICOLON","EXCLAMATION"])
check("عدد ۱۲،۵ بود.", ["عدد","۱۲،۵","بود"], [N,N,"PERIOD"])                                  # Persian digits
check("1,000 تومان.", ["1,000","تومان"], [N,"PERIOD"])                                        # number comma untouched
check("سلام ، دنیا .", ["سلام","دنیا"], ["COMMA","PERIOD"], exact=False)                      # space before punct
check("سلام،دنیا.", ["سلام","دنیا"], ["COMMA","PERIOD"], exact=False)                         # no space after
check("  سلام   دنیا.\n", ["سلام","دنیا"], [N,"PERIOD"])
check("(سلام) دنیا.", ["(سلام)","دنیا"], [N,"PERIOD"])
for s, why in [("واقعا؟!","multi_punct"),("صبر کن...","multi_punct"),("، سلام","leading_punct"),
               ("او گفت: «بیا.»","symbol_only_token"),("   ","empty"),("،","leading_punct"),
               ("سلام ، .","multi_punct"),("(سلام.)","symbol_only_token")]:
    assert P(s).reason == why, (s, P(s).reason, why)
assert P(None).reason == "not_a_string"
assert P("«سلام»").symbols == ("«","»")

# alignment with a mock tokenizer (word -> pieces of size<=3 chars)
def tokw(ws): return [[100+ord(w[k])%50 for k in range(0,len(w),3)] or [] for w in ws]
r=P("سلام، دوستان عزیزم؟")
enc,why=encode_words(r.words,r.labels,tokw,1,2,256)
assert why is None and enc["input_ids"][0]==1 and enc["input_ids"][-1]==2
lab=enc["labels"]; assert lab[0]==-100 and lab[-1]==-100
valid=[x for x in lab if x!=-100]; assert valid==r.labels
# label sits on the LAST piece of each word
pos=0; 
for w,l in zip(r.words,r.labels):
    k=len(tokw([w])[0]); assert lab[1+pos+k-1]==l and all(x==-100 for x in lab[1+pos:1+pos+k-1]); pos+=k
# truncation at word boundary keeps alignment
enc,_=encode_words(r.words,r.labels,tokw,1,2,6)
assert enc["truncated"] and len(enc["input_ids"])<=6 and [x for x in enc["labels"] if x!=-100]==r.labels[:enc["n_words"]]
assert encode_words(["x"*30],[0],tokw,1,2,6)[1]=="no_words_fit"
assert encode_words(["a",""],[0,0],tokw,1,2,50)[1]=="untokenizable_word"

# windows: full coverage, overlap, every word assigned
pc=[3]*100; w=plan_windows(pc,40,8); assert w[0][0]==0 and w[-1][1]==100
cov=set(); [cov.update(range(s,e)) for s,e in w]; assert cov==set(range(100))
b=best_window_for_each_word(100,w); assert all(w[b[i]][0]<=i<w[b[i]][1] for i in range(100))
assert len(plan_windows([1000,2,2],40))>=2
print("ALL CORE TESTS PASSED")


In [ ]:
# ---- tests with the REAL tokenizer ---------------------------------------------------------------------
def real_tokenize_words(words):
    return tok(list(words), add_special_tokens=False, return_attention_mask=False, return_token_type_ids=False)["input_ids"]

CASES = ["سلام، حال شما چطور است؟", "او گفت:«من می\u200cآیم».", "سیب, پرتقال, و موز.", "قیمت این کتاب 3.5 دلار است!",
         "نکته‌ی مهم این است: همیشه تلاش کن؛ هرگز تسلیم نشو."]
for s in CASES:
    r = parse_punctuated(s); assert r.reason is None, (s, r.reason)
    enc, why = encode_words(r.words, r.labels, real_tokenize_words, tok.cls_token_id, tok.sep_token_id, 10**6)
    assert why is None
    ids, lab = enc["input_ids"], enc["labels"]
    assert ids[0] == tok.cls_token_id and ids[-1] == tok.sep_token_id and lab[0] == lab[-1] == IGNORE_INDEX
    pos = 1
    for w, l in zip(r.words, r.labels):                    # the label must sit on the LAST piece of its word
        n = len(real_tokenize_words([w])[0]); assert n >= 1
        assert lab[pos + n - 1] == l and all(x == IGNORE_INDEX for x in lab[pos:pos + n - 1]); pos += n
    assert pos == len(ids) - 1
    assert [x for x in lab if x != IGNORE_INDEX] == r.labels
print("real-tokenizer alignment tests passed on", len(CASES), "sentences")

# truncation keeps alignment
r = parse_punctuated(" ".join(["کتاب"] * 40) + ".")
enc, _ = encode_words(r.words, r.labels, real_tokenize_words, tok.cls_token_id, tok.sep_token_id, 32)
assert enc["truncated"] and len(enc["input_ids"]) <= 32
assert [x for x in enc["labels"] if x != IGNORE_INDEX] == r.labels[:enc["n_words"]]
print("truncation test passed (kept", enc["n_words"], "of", len(r.words), "words)")

# agreement with the tokenizer's own pre-tokenised path (word_ids) on real validation sentences
if tok.is_fast:
    agree = total = unk = pieces_n = 0; first_diff = None
    for s in val_sentences[:500]:
        r = parse_punctuated(s)
        if r.reason: continue
        enc, why = encode_words(r.words, r.labels, real_tokenize_words, tok.cls_token_id, tok.sep_token_id, 10**6)
        if why: continue
        ref = tok(r.words, is_split_into_words=True)["input_ids"]
        total += 1; ok = ref == enc["input_ids"]; agree += ok
        if not ok and first_diff is None: first_diff = (r.words[:8], ref[:12], enc["input_ids"][:12])
        unk += sum(i == tok.unk_token_id for i in enc["input_ids"]); pieces_n += len(enc["input_ids"])
    print(f"per-word alignment == tokenizer is_split_into_words path on {agree}/{total} sentences; unk rate {unk / max(pieces_n, 1):.4%}")
    if agree < total:
        print("NOTE: differences do not affect correctness (training and inference both use the per-word path), first diff:", first_diff)
else:
    print("Slow tokenizer in use: word_ids() unavailable, the per-word alignment above is the verified procedure.")

## 7. Efficient data preparation
* The 16.9M-sentence training split is **streamed** (never fully in RAM, no pandas). The stream is shuffled with a seeded bounded buffer (`cfg.shuffle_buffer`), so the order is deterministic and a resume can fast-forward to the exact position.
* Pre-processing is done on bounded blocks of `buffer_steps × batch` sentences, which are then sorted by length into micro-batches (dynamic padding, little wasted compute) and shuffled in groups.
* Validation / test are only ever used in `evaluate_sentences`, which has no optimizer. Caching: the stream is deterministic and cheap to regenerate, so the cached artefacts are the downloaded validation/test files (`cache/hf_datasets`) and the data-integrity report (`evaluation/data_integrity.json`).

In [ ]:
class Featurizer:
    """sentence -> {input_ids, labels, n_words, truncated[, words]} with exclusion bookkeeping."""
    def __init__(self, tokenizer, max_length: int):
        self.tok, self.max_length, self.cache = tokenizer, max_length, {}
    def _fill(self, uniq: Iterable[str]) -> None:
        if len(self.cache) > 2_000_000:
            self.cache.clear()                                    # bound memory
        miss = [w for w in uniq if w not in self.cache]
        if miss:
            out = self.tok(miss, add_special_tokens=False, return_attention_mask=False, return_token_type_ids=False)["input_ids"]
            self.cache.update(zip(miss, out))
    def featurize(self, sentences: Sequence[str], keep_words: bool = False) -> Tuple[List[dict], Counter]:
        stats: Counter = Counter(); parsed = []; uniq = set()
        for s in sentences:
            stats["n_input"] += 1
            r = parse_punctuated(s)
            if r.reason:
                stats["excluded_" + r.reason] += 1; continue
            if r.symbols: stats["kept_with_unsupported_symbols"] += 1
            parsed.append(r); uniq.update(r.words)
        self._fill(uniq)
        feats = []
        for r in parsed:
            enc, why = encode_words(r.words, r.labels, lambda ws: [self.cache[w] for w in ws],
                                    self.tok.cls_token_id, self.tok.sep_token_id, self.max_length)
            if why:
                stats["excluded_" + why] += 1; continue
            stats["n_valid"] += 1; stats["n_words"] += enc["n_words"]; stats["n_truncated"] += int(enc["truncated"])
            if keep_words: enc["words"] = r.words[:enc["n_words"]]
            feats.append(enc)
        return feats, stats

featurizer = Featurizer(tok, cfg.max_length)

def collate(batch: Sequence[dict], pad_id: int) -> Dict[str, torch.Tensor]:
    B, L = len(batch), max(len(b["input_ids"]) for b in batch)
    ids = np.full((B, L), pad_id, np.int64); att = np.zeros((B, L), np.int64); lab = np.full((B, L), IGNORE_INDEX, np.int64)
    for i, b in enumerate(batch):
        n = len(b["input_ids"]); ids[i, :n] = b["input_ids"]; att[i, :n] = 1; lab[i, :n] = b["labels"]
    return {"input_ids": torch.from_numpy(ids), "attention_mask": torch.from_numpy(att), "labels": torch.from_numpy(lab)}

def make_stream(skip: int = 0):
    """Deterministic shuffled stream of the PUBLISHED train split (never merged with validation/test)."""
    ds = load_dataset(cfg.dataset_name, split="train", streaming=True)
    try:
        ds = ds.select_columns(["sentence"])
    except Exception:
        pass
    ds = ds.shuffle(seed=cfg.seed, buffer_size=cfg.shuffle_buffer)
    return ds.skip(skip) if skip else ds

In [ ]:
# ---- data-integrity report on a bounded sample of the (shuffled) train stream + the full val/test splits --------
integrity_path = DIRS["evaluation"] / "data_integrity.json"
if integrity_path.exists() and not cfg.force_integrity:
    integrity = json.loads(integrity_path.read_text()); print("Loaded cached integrity report:", integrity_path)
else:
    n = cfg.integrity_sample
    sample = [r["sentence"] for r in tqdm(itertools.islice(make_stream(), n), total=n, desc="sampling train stream")]
    feats, fst = featurizer.featurize(sample)
    seen, dups = set(), 0
    for s in sample:
        h = hashlib.sha1(normalize_text(s).encode()).hexdigest()
        dups += h in seen; seen.add(h)
    exact = spacing_only = bad = lex_bad = 0; sym = Counter(); chars, words_n = [], []
    for s in sample:
        r = parse_punctuated(s)
        if r.reason: continue
        rec = reconstruct(r.words, r.labels); ref = normalize_text(s)
        if rec == ref: exact += 1
        elif spacing_insensitive(rec) == spacing_insensitive(ref): spacing_only += 1
        else: bad += 1
        lex_bad += split_lexical(rec) != r.words
        sym.update(r.symbols); chars.append(len(s)); words_n.append(len(r.words))
    assert bad == 0, f"{bad} round-trip mismatches (beyond whitespace) - preprocessing is wrong"
    assert lex_bad == 0, "lexical tokens changed by reconstruction"
    lab_counts = np.zeros(C, np.int64); tlen = []
    for f in feats:
        l = np.asarray(f["labels"]); lab_counts += np.bincount(l[l != IGNORE_INDEX], minlength=C); tlen.append(len(f["input_ids"]))
    assert (lab_counts > 0).sum() > 1, "training labels contain a single class"
    # leakage checks that can be done by inspection (exact string match after whitespace normalisation)
    vset = {normalize_text(s) for s in val_sentences}; tset = {normalize_text(s) for s in get_test_sentences()}
    pct = lambda a, q: float(np.percentile(a, q)) if len(a) else None
    integrity = {
        "sampled_train_rows": n, "valid_examples": fst["n_valid"], "exclusion_rate": 1 - fst["n_valid"] / max(fst["n_input"], 1),
        "exclusions": {k: v for k, v in fst.items() if k.startswith("excluded_")}, "truncated_at_max_length": fst["n_truncated"],
        "truncation_rate": fst["n_truncated"] / max(fst["n_valid"], 1), "examples_with_unsupported_symbols": fst["kept_with_unsupported_symbols"],
        "top_unsupported_symbols": Counter(sym).most_common(15), "duplicate_rate_in_sample": dups / n,
        "label_counts": dict(zip(LABELS, map(int, lab_counts))), "label_share": dict(zip(LABELS, map(float, lab_counts / lab_counts.sum()))),
        "roundtrip": {"exact": exact, "spacing_normalised_only": spacing_only, "mismatch": bad, "lexical_changed": lex_bad},
        "chars_pct": {q: pct(chars, q) for q in (1, 50, 95, 99, 100)}, "words_pct": {q: pct(words_n, q) for q in (1, 50, 95, 99, 100)},
        "subword_len_pct": {q: pct(tlen, q) for q in (1, 50, 95, 99, 100)},
        "val_test_exact_overlap": len(vset & tset), "train_sample_in_val_or_test": sum(normalize_text(s) in vset or normalize_text(s) in tset for s in sample),
        "caveat": "Leakage is checked by exact string match on a bounded train sample only; the rest of the train split was not exhaustively compared.",
    }
    integrity_path.write_text(json.dumps(integrity, ensure_ascii=False, indent=2))
print(json.dumps(integrity, ensure_ascii=False, indent=1)[:3500])
tr = integrity["truncation_rate"]
print(f"\nmax_length={cfg.max_length}: {tr:.3%} of sampled sentences are truncated" + ("  -> consider a larger cfg.max_length" if tr > 0.01 else "  (acceptable)"))
print("Class imbalance (share):", {k: round(v, 4) for k, v in integrity["label_share"].items()})

## 8. Model initialization
`AlbertForTokenClassification` = pretrained ALBERT encoder + a **new** dropout+linear head (7 classes). The masked-LM head of the checkpoint is not used. The cell reports which weights were newly initialised / unused and **asserts that no encoder weight is missing**.

Below that, the shared utilities (metrics, evaluation, atomic checkpointing) are defined and exercised by a smoke test: forward pass, loss masking, gradients, a tiny overfit, checkpoint round-trip and end-to-end restoration.

In [ ]:
def build_model():
    m, info = AlbertForTokenClassification.from_pretrained(
        cfg.model_name, num_labels=C, id2label=ID2LABEL, label2id=LABEL2ID, output_loading_info=True)
    return m, info

model0, load_info = build_model()
print("newly initialised (expected: classifier only):", load_info["missing_keys"])
print("checkpoint weights not used (expected: MLM/SOP heads):", load_info["unexpected_keys"][:8], "..." if len(load_info["unexpected_keys"]) > 8 else "")
print("shape mismatches:", load_info["mismatched_keys"])
assert load_info["missing_keys"] and all(k.startswith("classifier") for k in load_info["missing_keys"]), \
    "Encoder weights were not loaded! (missing keys outside the classification head)"
assert not load_info["mismatched_keys"]
print(f"parameters: {sum(p.numel() for p in model0.parameters()) / 1e6:.2f}M | classifier_dropout_prob={model0.config.classifier_dropout_prob} | labels={model0.config.id2label}")
model0.to(DEVICE)

In [ ]:
# ----------------------------------------------------------------- metrics ---------------------------------------------
def metrics_from_cm(cm: np.ndarray) -> Dict[str, Any]:
    """Definitions (rows=true, cols=pred, valid word positions only):
    precision_c=TP/(pred=c)  recall_c=TP/(true=c)  F1_c=harmonic mean;
    macro_f1   = mean F1 over classes that occur in the reference;
    accuracy   = micro-F1 over all 7 classes (identical for single-label prediction);
    micro_f1_punct = micro-F1 over the 6 punctuation classes only (NONE excluded);
    insertion P/R/F1 = class-agnostic detection of 'a mark belongs here';
    PER = (substitutions + insertions + deletions) / number of reference marks."""
    cm = cm.astype(np.float64); tp = np.diag(cm); sup = cm.sum(1); pr = cm.sum(0)
    p = np.divide(tp, pr, out=np.zeros_like(tp), where=pr > 0); r = np.divide(tp, sup, out=np.zeros_like(tp), where=sup > 0)
    f = np.divide(2 * p * r, p + r, out=np.zeros_like(tp), where=(p + r) > 0)
    tpp, fpp, fnp = tp[1:].sum(), pr[1:].sum() - tp[1:].sum(), sup[1:].sum() - tp[1:].sum()
    mp = tpp / (tpp + fpp) if tpp + fpp else 0.0; mr = tpp / (tpp + fnp) if tpp + fnp else 0.0
    both, pp, tt = cm[1:, 1:].sum(), cm[:, 1:].sum(), cm[1:, :].sum()
    ip = both / pp if pp else 0.0; ir = both / tt if tt else 0.0
    return {"macro_f1": float(f[sup > 0].mean()), "accuracy": float(tp.sum() / cm.sum()), "micro_f1_all": float(tp.sum() / cm.sum()),
            "micro_f1_punct": float(2 * mp * mr / (mp + mr)) if mp + mr else 0.0,
            "per_class": {LABELS[i]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]), "support": int(sup[i])} for i in range(C)},
            "insertion_precision": float(ip), "insertion_recall": float(ir), "insertion_f1": float(2 * ip * ir / (ip + ir)) if ip + ir else 0.0,
            "punctuation_error_rate": float((cm.sum() - tp.sum()) / tt) if tt else None, "n_positions": int(cm.sum())}

@torch.no_grad()
def evaluate_sentences(model, sentences: Sequence[str], reconstruction: bool = True, max_errors: int = 25) -> Dict[str, Any]:
    """Token-level metrics from ACTUAL predictions on the valid positions; optional sentence-level reconstruction checks.
    Processes the sentences in bounded chunks, so the full 328k test split never sits in RAM as features."""
    was_training = model.training; model.eval(); dev = next(model.parameters()).device
    cm = np.zeros((C, C), np.int64); S: Counter = Counter(); loss_sum = 0.0; errors = []; t0 = time.time()
    for off in tqdm(range(0, len(sentences), cfg.eval_chunk), desc="evaluating", leave=False):
        feats, fst = featurizer.featurize(sentences[off:off + cfg.eval_chunk], keep_words=reconstruction); S.update(fst)
        order = sorted(range(len(feats)), key=lambda i: len(feats[i]["input_ids"]))
        for b in range(0, len(order), cfg.eval_bs):
            idx = order[b:b + cfg.eval_bs]; bt = collate([feats[i] for i in idx], tok.pad_token_id)
            lab = bt["labels"].to(dev)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
                logits = model(input_ids=bt["input_ids"].to(dev), attention_mask=bt["attention_mask"].to(dev)).logits
            logits = logits.float(); pred = logits.argmax(-1); mask = lab != IGNORE_INDEX
            loss_sum += F.cross_entropy(logits.view(-1, C), lab.view(-1), ignore_index=IGNORE_INDEX, reduction="sum").item()
            cm += torch.bincount(lab[mask] * C + pred[mask], minlength=C * C).view(C, C).cpu().numpy()
            if reconstruction:
                pc, lc = pred.cpu().numpy(), lab.cpu().numpy()
                for r_, i in enumerate(idx):
                    f = feats[i]
                    if f["truncated"]: S["recon_skipped_truncated"] += 1; continue
                    m = lc[r_] != IGNORE_INDEX; tr_, pr_ = lc[r_][m], pc[r_][m]; words = f["words"]
                    ok = bool((tr_ == pr_).all()); S["recon_examples"] += 1; S["exact_match"] += ok; S["baseline_exact_match"] += bool((tr_ == 0).all())
                    rec = reconstruct(words, pr_)
                    if split_lexical(rec) != words: S["lexical_changed"] += 1
                    if not ok and len(errors) < max_errors: errors.append({"reference": reconstruct(words, tr_), "prediction": rec})
    dt = time.time() - t0; res = metrics_from_cm(cm)
    base = np.zeros_like(cm); base[:, 0] = cm.sum(1); res["baseline_no_punctuation"] = {k: v for k, v in metrics_from_cm(base).items() if k != "per_class"}
    res.update(loss=loss_sum / max(cm.sum(), 1), confusion_matrix=cm.tolist(), counters=dict(S), seconds=dt,
               sentences_per_sec=S["n_valid"] / max(dt, 1e-9), n_sentences_requested=len(sentences),
               exclusion_rate=1 - S["n_valid"] / max(S["n_input"], 1), errors=errors)
    if reconstruction and S["recon_examples"]:
        n = S["recon_examples"]; res["exact_match_rate"] = S["exact_match"] / n; res["baseline_exact_match_rate"] = S["baseline_exact_match"] / n
        res["lexical_change_rate"] = S["lexical_changed"] / n
    if was_training: model.train()
    return res

# ----------------------------------------------------------------- atomic, verified saving -------------------------------
def atomic_write_json(path, obj) -> None:
    path = Path(path); tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2)); os.replace(tmp, path)

def verify_safetensors(model, path) -> None:
    sd = st_load_file(str(path)); msd = model.state_dict()
    missing = [n for n, _ in model.named_parameters() if n not in sd]
    assert not missing, f"saved file lacks parameters: {missing[:3]}"
    for k, v in sd.items():
        assert torch.equal(v, msd[k].detach().cpu()), f"saved tensor differs from memory: {k}"

def _swap_in(tmp: Path, final: Path) -> None:
    """Never delete the previous valid directory before the new one is complete and verified."""
    old = final.with_name(".old-" + final.name); shutil.rmtree(old, ignore_errors=True)
    if final.exists(): os.replace(final, old)
    os.replace(tmp, final); shutil.rmtree(old, ignore_errors=True)

def save_model_dir(model, tokenizer, dest, meta: Optional[dict] = None) -> Path:
    dest = Path(dest); tmp = dest.with_name(".tmp-" + dest.name); shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
    model.save_pretrained(tmp, safe_serialization=True); tokenizer.save_pretrained(tmp)
    (tmp / "label_mappings.json").write_text(json.dumps({"id2label": ID2LABEL, "label2id": LABEL2ID, "label2punct": LABEL2PUNCT}, ensure_ascii=False, indent=2))
    if meta is not None: (tmp / "training_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2))
    verify_safetensors(model, tmp / "model.safetensors"); _swap_in(tmp, dest); return dest

def get_rng_state() -> dict:
    return {"py": random.getstate(), "np": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if HAVE_CUDA else None}
def set_rng_state(s: dict) -> None:
    random.setstate(s["py"]); np.random.set_state(s["np"]); torch.set_rng_state(s["torch"])
    if HAVE_CUDA and s.get("cuda") is not None: torch.cuda.set_rng_state_all(s["cuda"])

def is_valid_ckpt(p: Path) -> bool:
    return p.is_dir() and (p / "CHECKPOINT_OK").exists() and (p / "model.safetensors").exists() and (p / "trainer_state.json").exists()
def find_latest_checkpoint(root: Path) -> Optional[Path]:
    c = sorted([p for p in Path(root).glob("step-*") if is_valid_ckpt(p)], reverse=True)   # partial writes live in .tmp-* and are ignored
    return c[0] if c else None

def save_checkpoint(model, opt, sched, scaler, state: dict, root: Path, tag_meta: dict) -> Path:
    root = Path(root); name = f"step-{state['global_step']:09d}"; tmp = root / (".tmp-" + name); shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
    model.save_pretrained(tmp, safe_serialization=True)
    torch.save(opt.state_dict(), tmp / "optimizer.pt"); torch.save(sched.state_dict(), tmp / "scheduler.pt")
    torch.save(scaler.state_dict(), tmp / "scaler.pt"); torch.save(get_rng_state(), tmp / "rng_state.pt")
    (tmp / "trainer_state.json").write_text(json.dumps(state, ensure_ascii=False, indent=2))
    verify_safetensors(model, tmp / "model.safetensors")
    digest = hashlib.sha256((tmp / "model.safetensors").read_bytes()).hexdigest()
    (tmp / "CHECKPOINT_OK").write_text(json.dumps({"step": state["global_step"], "model_sha256": digest, **tag_meta}))   # marker written LAST
    final = root / name; _swap_in(tmp, final)
    atomic_write_json(root / "latest.json", {"latest": name, "step": state["global_step"]})
    return final

def prune_checkpoints(root: Path, keep_last: int) -> None:
    cks = sorted([p for p in Path(root).glob("step-*") if is_valid_ckpt(p)])
    for p in cks[:-keep_last]: shutil.rmtree(p, ignore_errors=True); print("deleted old checkpoint", p.name)

In [ ]:
# ----------------------------------------------------------------- smoke test -----------------------------------------------
SMOKE = ["سلام، حال شما چطور است؟", "او گفت که فردا می\u200cآید.", "سیب، پرتقال، موز و انگور را خریدم.", "آیا فردا به دانشگاه می\u200cروی؟",
         "لطفاً درها را ببندید؛ چراغ\u200cها را هم خاموش کنید.", "نکته مهم این است: همیشه تلاش کن.", "وای چه منظرهٔ زیبایی!", "من امروز کتاب خواندم.",
         "او پرسید: چرا دیر آمدی؟", "هوا امروز سرد است، اما آفتابی است.", "ساعت 10:30 جلسه شروع می\u200cشود.", "دانش آموزان، معلمان و والدین آمدند."]
sf, sst = featurizer.featurize(SMOKE, keep_words=True)
assert len(sf) == len(SMOKE), f"smoke sentences unexpectedly excluded: {dict(sst)}"
bt = collate(sf, tok.pad_token_id)
sm = copy.deepcopy(model0).train()                        # keep model0 pristine
ids, att, lab = bt["input_ids"].to(DEVICE), bt["attention_mask"].to(DEVICE), bt["labels"].to(DEVICE)
out = sm(input_ids=ids, attention_mask=att); assert out.logits.shape == (len(sf), ids.shape[1], C)
nvalid = int((lab != IGNORE_INDEX).sum()); assert nvalid == sum(f["n_words"] for f in sf), "valid label count != number of words"
loss = F.cross_entropy(out.logits.float().view(-1, C), lab.view(-1), ignore_index=IGNORE_INDEX, reduction="sum") / nvalid
assert torch.isfinite(loss); loss.backward()
assert sm.classifier.weight.grad.abs().sum() > 0 and sm.albert.embeddings.word_embeddings.weight.grad.abs().sum() > 0, "no gradient reaches head/encoder"
# loss masking: arbitrary logits at ignored positions must not change the loss
noisy = out.logits.detach().float().clone(); noisy[lab == IGNORE_INDEX] += 123.0
l2 = F.cross_entropy(noisy.view(-1, C), lab.view(-1), ignore_index=IGNORE_INDEX, reduction="sum") / nvalid
assert torch.allclose(l2, loss.detach(), atol=1e-4), "ignored positions leak into the loss"
sm.zero_grad(set_to_none=True)
# tiny overfit: loss must drop clearly
o = torch.optim.AdamW(sm.parameters(), lr=1e-4); first = None
for i in range(40):
    o.zero_grad(); lg = sm(input_ids=ids, attention_mask=att).logits.float()
    l = F.cross_entropy(lg.view(-1, C), lab.view(-1), ignore_index=IGNORE_INDEX); l.backward(); o.step()
    first = first if first is not None else l.item()
print(f"smoke: loss {first:.3f} -> {l.item():.3f} over 40 steps on {len(sf)} sentences")
assert l.item() < 0.7 * first, "model cannot even overfit 12 sentences - check labels/alignment"
# checkpoint round trip in a FRESH model instance + end-to-end inference
SMOKE_DIR = Path("/content/smoke_model"); save_model_dir(sm, tok, SMOKE_DIR, {"smoke": True})
sm.eval(); fresh = AlbertForTokenClassification.from_pretrained(SMOKE_DIR).to(DEVICE).eval()
with torch.no_grad():
    a = sm(input_ids=ids, attention_mask=att).logits.float(); b = fresh(input_ids=ids, attention_mask=att).logits.float()
assert torch.allclose(a, b, atol=1e-4), "reloaded checkpoint gives different logits"
smoke_restorer = PunctuationRestorer(str(SMOKE_DIR), device=str(DEVICE))
for s in SMOKE[:4]:
    stripped = " ".join(split_lexical(s)); print("  ", stripped, "->", smoke_restorer.restore(stripped))
print("SMOKE TEST PASSED (forward/backward, masking, gradients, overfit, checkpoint reload, end-to-end restore)")
del sm, fresh, o, smoke_restorer; torch.cuda.empty_cache()

## 9. Pilot training
The same engine (`train_loop`) is used for the pilot and for the resumable run, so the pilot validates the real code path.

**Engine design**
* Stream → block of `buffer_steps × train_bs × grad_accum` sentences → featurise → sort by length → micro-batches → groups of `grad_accum` → groups shuffled (seeded). Dynamic padding, no wasted `max_length` padding.
* Loss = summed token cross-entropy over *valid* positions of an accumulation group ÷ number of valid positions in the group (exact token-level weighting across micro-batches). AdamW (no decay on bias/LayerNorm), linear warm-up/decay, fp16 autocast + `GradScaler`, gradient clipping.
* Checkpoints and evaluations happen only at block boundaries, so the saved stream position (`examples_read`) is exact. Resume restores model, optimizer, scheduler, scaler, RNG states, counters, best-metric/early-stopping state, and fast-forwards the seeded stream. Examples are not repeated within a pass.
* A CUDA OOM skips the group, frees memory, and raises a clear message if it repeats (`cfg.max_oom_skips`); non-finite losses are skipped and counted.
* Pilot caveat: the pilot sees the first block of the same seeded stream as the main run, its weights are discarded.

In [ ]:
@dataclass
class RunSpec:
    name: str; ckpt_dir: Path; log_path: Path; max_steps: int; eval_steps: int; save_steps: int; eval_limit: int
    save_checkpoints: bool = True; save_best: bool = True; patience: int = 0; resume: str = "auto"

def new_state(run: RunSpec) -> dict:
    return {"run": run.name, "global_step": 0, "examples_read": 0, "examples_used": 0, "best_metric": None, "best_step": None,
            "bad_evals": 0, "stopped_early": False, "train_seconds": 0.0, "sessions": 0, "oom_skips": 0, "nonfinite_skips": 0,
            "label_counts": [0] * C, "exclusions": {}, "n_valid_train": 0, "max_steps": run.max_steps, "seed": cfg.seed,
            "examples_per_step": cfg.train_bs * cfg.grad_accum, "buffer_steps": cfg.buffer_steps, "stream_state": None,
            "last_val": None, "completed": False, "exhausted_stream": False}

def train_loop(run: RunSpec, val_sents: Sequence[str]):
    seed_everything(cfg.seed); root = Path(run.ckpt_dir); root.mkdir(parents=True, exist_ok=True); Path(run.log_path).parent.mkdir(parents=True, exist_ok=True)
    ck = None
    if run.resume == "auto": ck = find_latest_checkpoint(root)
    elif run.resume in (None, "none"):
        if find_latest_checkpoint(root) and not cfg.allow_fresh_start:
            raise RuntimeError(f"Valid checkpoints exist in {root}. Use cfg.resume='auto' to continue, or set cfg.allow_fresh_start=True to deliberately start over.")
    else:
        ck = Path(run.resume); assert is_valid_ckpt(ck), f"{ck} is not a complete checkpoint"
    if ck:
        st = json.loads((ck / "trainer_state.json").read_text()); model = AlbertForTokenClassification.from_pretrained(ck)
        assert st["seed"] == cfg.seed and st["examples_per_step"] == cfg.train_bs * cfg.grad_accum and st["buffer_steps"] == cfg.buffer_steps, \
            "seed / effective batch / buffer_steps differ from the checkpoint - data progress would no longer be exact"
        if st["max_steps"] != run.max_steps:
            assert cfg.allow_schedule_change, f"max_train_steps changed ({st['max_steps']} -> {run.max_steps}); set cfg.allow_schedule_change=True to extend"
            st["max_steps"] = run.max_steps
        st["stopped_early"] = False if cfg.early_stopping_patience == 0 else st["stopped_early"]
        print(f"RESUMING from {ck.name}: step {st['global_step']}, {st['examples_read']:,} stream rows already consumed")
    else:
        model, _ = build_model(); st = new_state(run); print("Starting from the pretrained encoder (step 0)")
    model.to(DEVICE).train(); st["sessions"] += 1
    nd = ("bias", "LayerNorm.weight")
    groups_ = [{"params": [p for n, p in model.named_parameters() if not any(k in n for k in nd)], "weight_decay": cfg.weight_decay},
               {"params": [p for n, p in model.named_parameters() if any(k in n for k in nd)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups_, lr=cfg.lr)
    sched = get_linear_schedule_with_warmup(opt, int(cfg.warmup_ratio * run.max_steps), run.max_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)
    if ck:
        opt.load_state_dict(torch.load(ck / "optimizer.pt", map_location="cpu")); sched.load_state_dict(torch.load(ck / "scheduler.pt"))
        scaler.load_state_dict(torch.load(ck / "scaler.pt")); set_rng_state(torch.load(ck / "rng_state.pt", weights_only=False))  # our own trusted file
    # ---- data stream (exact position) ----
    if cfg.resume_mode == "state_dict" and st.get("stream_state"):
        stream = make_stream(); stream.load_state_dict(st["stream_state"]); print("stream restored from datasets state_dict")
    else:
        if st["examples_read"]: print(f"fast-forwarding the seeded stream by {st['examples_read']:,} rows (one-off cost)...")
        stream = make_stream(skip=st["examples_read"])
    it = iter(stream); need = cfg.buffer_steps * cfg.train_bs * cfg.grad_accum
    if HAVE_CUDA: torch.cuda.reset_peak_memory_stats()
    t_sess = time.time(); last_eval = last_save = st["global_step"]; w_loss = torch.zeros((), device=DEVICE); w_steps = w_ex = 0; w_t = time.time(); ex_s_samples = []; consec_bad = 0
    pbar = tqdm(total=run.max_steps, initial=st["global_step"], desc=run.name)
    while st["global_step"] < run.max_steps and not st["stopped_early"]:
        t_blk = time.time(); raw = []; exhausted = True
        for row in it:
            raw.append(row["sentence"])
            if len(raw) >= need: exhausted = False; break
        if not raw: st["exhausted_stream"] = True; break
        st["examples_read"] += len(raw)
        feats, fst = featurizer.featurize(raw); st["n_valid_train"] += fst["n_valid"]
        for k, v in fst.items():
            if k.startswith("excluded_"): st["exclusions"][k] = st["exclusions"].get(k, 0) + v
        feats.sort(key=lambda f: len(f["input_ids"]))
        mbs = [feats[i:i + cfg.train_bs] for i in range(0, len(feats) - cfg.train_bs + 1, cfg.train_bs)]
        grps = [mbs[i:i + cfg.grad_accum] for i in range(0, len(mbs) - cfg.grad_accum + 1, cfg.grad_accum)]
        random.Random(cfg.seed * 1_000_003 + st["global_step"]).shuffle(grps); grps = grps[: run.max_steps - st["global_step"]]
        for g in grps:
            batches = [collate(mb, tok.pad_token_id) for mb in g]; nvalid = sum(int((b["labels"] != IGNORE_INDEX).sum()) for b in batches)
            try:
                tot = torch.zeros((), device=DEVICE)
                for b in batches:
                    with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
                        lg = model(input_ids=b["input_ids"].to(DEVICE), attention_mask=b["attention_mask"].to(DEVICE)).logits
                    loss = F.cross_entropy(lg.float().view(-1, C), b["labels"].to(DEVICE).view(-1), ignore_index=IGNORE_INDEX, reduction="sum") / nvalid
                    scaler.scale(loss).backward(); tot += loss.detach()
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                if not torch.isfinite(tot):                       # fp16 overflow in grads is handled by the scaler; a non-finite LOSS is a real problem
                    opt.zero_grad(set_to_none=True); st["nonfinite_skips"] += 1; consec_bad += 1
                    if consec_bad > 20: raise RuntimeError("20 consecutive non-finite losses: lower cfg.lr or disable cfg.fp16")
                    continue
                consec_bad = 0; scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            except torch.cuda.OutOfMemoryError:
                opt.zero_grad(set_to_none=True); del batches; torch.cuda.empty_cache(); st["oom_skips"] += 1
                if st["oom_skips"] > cfg.max_oom_skips:
                    raise RuntimeError("Repeated CUDA OOM: lower cfg.train_bs (and raise cfg.grad_accum to keep the effective batch) or lower cfg.max_length.")
                continue
            st["global_step"] += 1; n_ex = len(g) * cfg.train_bs; st["examples_used"] += n_ex; w_ex += n_ex; w_steps += 1; w_loss += tot; pbar.update(1)
            for b in batches:
                l_ = b["labels"].numpy(); st["label_counts"] = (np.asarray(st["label_counts"]) + np.bincount(l_[l_ != IGNORE_INDEX], minlength=C)).tolist()
            if st["global_step"] % cfg.logging_steps == 0:
                dt = time.time() - w_t; rec = {"step": st["global_step"], "train_loss": float(w_loss.item() / w_steps), "lr": sched.get_last_lr()[0],
                       "examples_per_sec": w_ex / dt, "peak_gpu_gb": torch.cuda.max_memory_allocated() / 2**30 if HAVE_CUDA else 0.0,
                       "examples_read": st["examples_read"], "grad_scale": float(scaler.get_scale()) if USE_FP16 else 1.0, "time": time.time()}
                with open(run.log_path, "a") as fh: fh.write(json.dumps(rec) + "\n")
                ex_s_samples.append(rec["examples_per_sec"]); pbar.set_postfix(loss=f"{rec['train_loss']:.4f}", ex_s=f"{rec['examples_per_sec']:.0f}")
                w_loss.zero_(); w_steps = w_ex = 0; w_t = time.time()
        # ---------------- block boundary: stream position is exact here ----------------
        if cfg.resume_mode == "state_dict":
            try: st["stream_state"] = stream.state_dict()
            except Exception: st["stream_state"] = None
        finished = st["global_step"] >= run.max_steps or exhausted
        out_of_time = bool(cfg.session_time_budget_min) and (time.time() - t_sess) / 60 >= cfg.session_time_budget_min
        if run.eval_limit and (finished or st["global_step"] - last_eval >= run.eval_steps):
            m = evaluate_sentences(model, list(val_sents[:run.eval_limit]), reconstruction=False); last_eval = st["global_step"]; model.train()
            st["last_val"] = {k: m[k] for k in ("macro_f1", "micro_f1_punct", "accuracy", "loss")}
            with open(run.log_path, "a") as fh: fh.write(json.dumps({"step": st["global_step"], "val_macro_f1": m["macro_f1"], "val_micro_f1_punct": m["micro_f1_punct"],
                    "val_accuracy": m["accuracy"], "val_loss": m["loss"], "val_sentences": m["counters"].get("n_valid", 0)}) + "\n")
            print(f"[step {st['global_step']}] val macro-F1 {m['macro_f1']:.4f} | punct micro-F1 {m['micro_f1_punct']:.4f} | loss {m['loss']:.4f}")
            if st["best_metric"] is None or m["macro_f1"] > st["best_metric"]:
                st["best_metric"], st["best_step"], st["bad_evals"] = m["macro_f1"], st["global_step"], 0
                if run.save_best: save_model_dir(model, tok, root / "best", {"step": st["global_step"], "val_macro_f1": m["macro_f1"], "val_limit": run.eval_limit})
            else:
                st["bad_evals"] += 1
                if run.patience and st["bad_evals"] >= run.patience: st["stopped_early"] = True; print(f"early stopping: no macro-F1 gain in {run.patience} evaluations")
        st["train_seconds"] += time.time() - t_blk
        if run.save_checkpoints and (finished or out_of_time or st["stopped_early"] or st["global_step"] - last_save >= run.save_steps):
            if finished: st["completed"] = st["global_step"] >= run.max_steps
            p = save_checkpoint(model, opt, sched, scaler, st, root, {"run": run.name}); last_save = st["global_step"]; print("checkpoint saved:", p.name)
            if cfg.delete_old_checkpoints: prune_checkpoints(root, cfg.keep_last)
        if out_of_time: print(f"session time budget reached at step {st['global_step']} - checkpoint saved, re-run this cell to continue."); break
        if exhausted: st["exhausted_stream"] = True; break
    pbar.close(); st["completed"] = st["global_step"] >= run.max_steps
    st["median_examples_per_sec"] = float(np.median(ex_s_samples)) if ex_s_samples else None
    st["peak_gpu_gb"] = torch.cuda.max_memory_allocated() / 2**30 if HAVE_CUDA else 0.0
    return model, st

In [ ]:
# ---- pilot run (skipped automatically when its result already exists) -----------------------------------------
pilot_dir = DIRS["logs"] / "pilot"; pilot_json = pilot_dir / "pilot_result.json"
if cfg.run_pilot and (cfg.force_pilot or not pilot_json.exists()):
    if (pilot_dir / "pilot_log.jsonl").exists(): (pilot_dir / "pilot_log.jsonl").unlink()
    run = RunSpec("pilot", pilot_dir / "ckpt", pilot_dir / "pilot_log.jsonl", max_steps=cfg.pilot_steps, eval_steps=cfg.pilot_steps, save_steps=10**9,
                  eval_limit=cfg.pilot_eval_limit, save_checkpoints=False, save_best=False, patience=0, resume="none")
    pmodel, pst = train_loop(run, val_sentences)
    ex_s = pst["median_examples_per_sec"] or 0.0; full_h = EXPECTED_SPLITS["train"] / ex_s / 3600 if ex_s else None
    pilot = {"steps": pst["global_step"], "examples_used": pst["examples_used"], "median_examples_per_sec": ex_s, "peak_gpu_gb": pst["peak_gpu_gb"],
             "gpu_total_gb": GPU_INFO.get("total_gb"), "val": pst["last_val"], "nonfinite_skips": pst["nonfinite_skips"], "oom_skips": pst["oom_skips"],
             "estimated_hours_for_main_run": cfg.max_train_steps * cfg.train_bs * cfg.grad_accum / ex_s / 3600 if ex_s else None,
             "estimated_hours_one_full_epoch": full_h, "note": "Estimates extrapolate pilot throughput; they are not guarantees."}
    pilot_dir.mkdir(parents=True, exist_ok=True); atomic_write_json(pilot_json, pilot); del pmodel; torch.cuda.empty_cache()
if pilot_json.exists():
    pilot = json.loads(pilot_json.read_text()); print(json.dumps(pilot, indent=1))
    if pilot["peak_gpu_gb"] and GPU_INFO.get("total_gb") and pilot["peak_gpu_gb"] > 0.85 * GPU_INFO["total_gb"]:
        print("WARNING: peak memory is close to the GPU limit; consider cfg.train_bs=8, cfg.grad_accum=4.")
    assert pilot["nonfinite_skips"] == 0 or pilot["nonfinite_skips"] < 5, "numerical instability in the pilot"

## 10. Resumable fine-tuning
Run this cell, let it train; if Colab disconnects, reconnect, re-run sections 2-8 (cheap) and this cell - it finds the newest **complete** checkpoint (marker file written last, weights verified after saving) and continues exactly where it stopped.

* **Continue / extend:** to train longer, raise `cfg.max_train_steps` and set `cfg.allow_schedule_change=True` (the LR schedule is recomputed, so the LR can jump up a little). Otherwise a finished run simply reloads its last checkpoint.
* **Colab time limits:** set `cfg.session_time_budget_min` (e.g. 330) so the loop saves and stops cleanly before the session is reclaimed.
* **Start over deliberately:** `cfg.resume="none"` *and* `cfg.allow_fresh_start=True`. Nothing is ever deleted unless `cfg.delete_old_checkpoints=True`.
* A model is only called *fully trained* in the final report if `examples_read` reaches the whole published training split.

In [ ]:
MAIN_LOG = DIRS["logs"] / "train_log.jsonl"
if cfg.run_training:
    run = RunSpec("main", DIRS["checkpoints"], MAIN_LOG, max_steps=cfg.max_train_steps, eval_steps=cfg.eval_steps, save_steps=cfg.save_steps,
                  eval_limit=cfg.val_limit_during_training, save_checkpoints=True, save_best=True, patience=cfg.early_stopping_patience, resume=cfg.resume)
    model, train_state = train_loop(run, val_sentences)
    atomic_write_json(DIRS["logs"] / "train_state_summary.json", {k: v for k, v in train_state.items() if k != "stream_state"})
    print(json.dumps({k: train_state[k] for k in ("global_step", "examples_read", "examples_used", "best_metric", "best_step", "completed",
                                                   "stopped_early", "oom_skips", "nonfinite_skips", "train_seconds")}, indent=1))
    frac = train_state["examples_read"] / EXPECTED_SPLITS["train"]
    print(f"Stream rows consumed: {train_state['examples_read']:,} = {frac:.2%} of the published train split -> " + ("FULL pass" if frac >= 1 else "SUBSET-trained model"))
else:
    print("cfg.run_training=False: skipped (existing checkpoints, if any, are used below).")

## 11. Validation and checkpoint selection
**Criterion:** the checkpoint with the highest **macro-F1 on the validation split** (evaluated on `val_limit_during_training` sentences during training; saved as `checkpoints/best`). Here the selected model is re-scored on the **full** validation split. The test split is not touched until section 12.

In [ ]:
best_dir = DIRS["checkpoints"] / "best"
if not (best_dir / "model.safetensors").exists():
    last = find_latest_checkpoint(DIRS["checkpoints"])
    assert last is not None, "No trained model found. Run section 10 first."
    print("No 'best' directory yet - using latest checkpoint", last.name); best_dir = last
final_model = AlbertForTokenClassification.from_pretrained(best_dir).to(DEVICE).eval()
val_res = evaluate_sentences(final_model, val_sentences, reconstruction=True)
val_res["model_dir"] = str(best_dir); val_res["subset"] = False
atomic_write_json(DIRS["evaluation"] / "validation_metrics.json", {k: v for k, v in val_res.items() if k != "errors"})
print(f"validation (full, {val_res['counters']['n_valid']:,} usable of {val_res['counters']['n_input']:,}): macro-F1 {val_res['macro_f1']:.4f} | punct micro-F1 {val_res['micro_f1_punct']:.4f} | acc {val_res['accuracy']:.4f}")
print("baseline (no punctuation): macro-F1 %.4f | acc %.4f" % (val_res["baseline_no_punctuation"]["macro_f1"], val_res["baseline_no_punctuation"]["accuracy"]))
print("training curve:")
if MAIN_LOG.exists():
    log = pd.read_json(MAIN_LOG, lines=True)
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
    if "train_loss" in log: log.dropna(subset=["train_loss"]).plot(x="step", y="train_loss", ax=ax[0], title="train loss")
    if "val_macro_f1" in log: log.dropna(subset=["val_macro_f1"]).plot(x="step", y="val_macro_f1", marker="o", ax=ax[1], title="validation macro-F1")
    if "examples_per_sec" in log: log.dropna(subset=["examples_per_sec"]).plot(x="step", y="examples_per_sec", ax=ax[2], title="examples / s")
    plt.tight_layout(); fig.savefig(DIRS["evaluation"] / "training_curves.png", dpi=120); plt.show()

## 12. Final test evaluation
Run once, after model selection. `cfg.test_limit=None` evaluates the **full published test split**; an integer evaluates only the first *n* sentences and the report then says "subset". Sentences that cannot be represented by the label scheme (section 5) are excluded and their rate is reported.

In [ ]:
test_all = get_test_sentences(); test_sents = test_all if cfg.test_limit is None else test_all[:cfg.test_limit]
test_res = evaluate_sentences(final_model, test_sents, reconstruction=True)
test_res["subset"] = cfg.test_limit is not None and cfg.test_limit < len(test_all); test_res["n_test_split"] = len(test_all)
test_res["model_dir"] = str(best_dir)
print(("SUBSET of the test split: %d / %d sentences" % (len(test_sents), len(test_all))) if test_res["subset"] else f"FULL test split: {len(test_all):,} sentences")
print(f"usable sentences {test_res['counters']['n_valid']:,} | excluded {test_res['exclusion_rate']:.3%} {({k:v for k,v in test_res['counters'].items() if k.startswith('excluded_')})}")
hdr = ["macro_f1", "micro_f1_punct", "accuracy", "insertion_precision", "insertion_recall", "insertion_f1", "punctuation_error_rate", "exact_match_rate", "lexical_change_rate"]
base = test_res["baseline_no_punctuation"]
print(pd.DataFrame({"model": {k: test_res.get(k) for k in hdr}, "baseline: no punctuation": {**{k: base.get(k) for k in hdr}, "exact_match_rate": test_res.get("baseline_exact_match_rate"), "lexical_change_rate": 0.0}}).round(4).to_string())
assert test_res["lexical_change_rate"] == 0.0, "lexical content changed during reconstruction - this must be fixed, not accepted"
print("\nPer-class report (valid word positions):")
pc = pd.DataFrame(test_res["per_class"]).T; print(pc.round(4).to_string())
cm = np.array(test_res["confusion_matrix"]); y_t = np.repeat(np.arange(C * C) // C, cm.ravel()); y_p = np.repeat(np.arange(C * C) % C, cm.ravel())
from sklearn.metrics import classification_report
print("\n", classification_report(y_t, y_p, labels=list(range(C)), target_names=LABELS, digits=4, zero_division=0))
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, mat, ttl in ((ax[0], cm, "counts"), (ax[1], cm / np.maximum(cm.sum(1, keepdims=True), 1), "row-normalised (recall)")):
    a.imshow(mat, cmap="Blues"); a.set_xticks(range(C)); a.set_yticks(range(C)); a.set_xticklabels(LABELS, rotation=45, ha="right"); a.set_yticklabels(LABELS)
    a.set_xlabel("predicted"); a.set_ylabel("true"); a.set_title("Test confusion matrix - " + ttl)
    for i in range(C):
        for j in range(C): a.text(j, i, f"{mat[i, j]:.2f}" if mat.dtype.kind == "f" else int(mat[i, j]), ha="center", va="center", fontsize=7)
plt.tight_layout(); fig.savefig(DIRS["evaluation"] / "confusion_matrix_test.png", dpi=130); plt.show()
pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(DIRS["evaluation"] / "confusion_matrix_test.csv")
print("Sample sentence-level errors (reference vs prediction):")
for e in test_res["errors"][:8]: print("  REF :", e["reference"], "\n  PRED:", e["prediction"], "\n")

In [ ]:
# ---- inference latency / throughput (tokenisation + model + reconstruction), on the selected checkpoint -----------------
restorer = PunctuationRestorer(str(best_dir), device=str(DEVICE))
lat_src = [" ".join(split_lexical(s)) for s in test_all[:300] if parse_punctuated(s).reason is None][:200]
for s in lat_src[:5]: restorer.restore(s)                                   # warm-up
t = []
for s in lat_src:
    t0 = time.perf_counter(); restorer.restore(s)
    if HAVE_CUDA: torch.cuda.synchronize()
    t.append(time.perf_counter() - t0)
t0 = time.perf_counter(); restorer.restore(lat_src)
if HAVE_CUDA: torch.cuda.synchronize()
tb = time.perf_counter() - t0
latency = {"n_sentences": len(lat_src), "single_median_ms": float(np.median(t) * 1e3), "single_p95_ms": float(np.percentile(t, 95) * 1e3),
           "single_sentences_per_sec": len(t) / sum(t), "batched_sentences_per_sec": len(lat_src) / tb, "device": str(DEVICE), "fp16": restorer.fp16}
test_res["latency"] = latency; print(latency)
atomic_write_json(DIRS["evaluation"] / "test_metrics.json", {k: v for k, v in test_res.items() if k != "errors"})
atomic_write_json(DIRS["evaluation"] / "test_errors_sample.json", test_res["errors"])

## 13. Inference and reconstruction
`PunctuationRestorer.restore(text | list_of_texts)` returns the restored string(s).

* Input is reduced to its words with the *same parser as training*; any punctuation already in the input is discarded and re-predicted (numbers such as `3.5`, `10:30` are kept intact). Whitespace is normalised.
* **Lexical preservation** is enforced on every call (`verify=True`): the words of the output must equal the words of the input, otherwise a `RuntimeError` is raised.
* **Long inputs are never truncated.** Words are split into overlapping windows that fit `max_length` sub-words (16 words of overlap); every word takes its prediction from the window where it is farthest from a window edge, so there is no forced punctuation at segment boundaries. A single word longer than a window is clipped for context only.
* A confidence threshold is intentionally **not** exposed: it was not validated here, and an unvalidated knob would be misleading.

In [ ]:
EXAMPLES = ["سلام حال شما چطور است", "آیا فردا به دانشگاه می\u200cروی", "من سیب پرتقال موز و انگور خریدم", "او گفت که فردا نمی\u200cآید اما من مطمئن نیستم",
            "چه ساعتی به خانه برمی\u200cگردی", "لطفا درها را ببندید و چراغ\u200cها را خاموش کنید", "ساعت 10:30 جلسه شروع می\u200cشود",
            "وای چه منظره زیبایی", "قیمت این کتاب 3.5 دلار است", "او پرسید چرا دیر آمدی من گفتم ترافیک بود", "سلام",
            "سلام، دنیا.", "   ", "", "پدر مادر برادر و خواهرم همگی در این مهمانی حضور داشتند"]
single = [restorer.restore(x) for x in EXAMPLES]; batch = restorer.restore(EXAMPLES)
assert single == batch, "batched and single-sentence results differ"
for x, y in zip(EXAMPLES, batch):
    assert split_lexical(x) == split_lexical(y), f"lexical change for {x!r}"           # independent re-check of lexical preservation
    print(f"{x!r:75} -> {y!r}")
assert restorer.restore("") == "" and restorer.restore("   ") == ""
# long input: ~3,000 words, must come back complete, in order, with punctuation inserted
long_src = " ".join(" ".join(split_lexical(s)) for s in val_sentences[:150]); long_words = split_lexical(long_src)
counts = [len(x) for x in restorer._pieces(long_words)]; wins = plan_windows([max(c, 1) for c in counts], restorer.max_length - 2, restorer.overlap)
t0 = time.time(); long_out = restorer.restore(long_src)
assert split_lexical(long_out) == long_words, "long-input restoration changed lexical content"
print(f"long input: {len(long_words)} words, {len(wins)} windows, {time.time() - t0:.2f}s, {sum(ch in '،.؟!؛:' for ch in long_out)} marks inserted")
# lexical-preservation test on real held-out sentences
chk = 0
for s in val_sentences[:300]:
    r = parse_punctuated(s)
    if r.reason: continue
    out = restorer.restore(" ".join(r.words)); assert split_lexical(out) == r.words; chk += 1
print(f"lexical preservation verified on {chk} held-out validation sentences")
atomic_write_json(DIRS["examples"] / "inference_examples.json", [{"input": x, "output": y} for x, y in zip(EXAMPLES, batch)])

## 14. Save and verify the final model
The selected model is written atomically (verified `safetensors`, tokenizer, config, `id2label`/`label2id`, training metadata) to `model/` together with the inference code, configuration, a README, and checksums. Verification then runs in a **fresh Python process** whose working directory and import path contain only the Drive model folder, so nothing that exists only in temporary Colab storage can be used.

In [ ]:
summary = json.loads((DIRS["logs"] / "train_state_summary.json").read_text()) if (DIRS["logs"] / "train_state_summary.json").exists() else {}
frac = summary.get("examples_read", 0) / EXPECTED_SPLITS["train"]
meta = {"base_model": cfg.model_name, "dataset": cfg.dataset_name, "labels": LABELS, "max_length": cfg.max_length,
        "selected_from": str(best_dir), "train_examples_read": summary.get("examples_read"), "train_fraction": frac,
        "scope": "FULL one pass of the published train split" if frac >= 1 else "SUBSET of the published train split",
        "global_step": summary.get("global_step"), "best_step": summary.get("best_step"), "val_macro_f1_selection": summary.get("best_metric")}
save_model_dir(final_model, tok, DIRS["model"], meta)
for f in ("punc_core.py", "punc_inference.py"): shutil.copy(PKG / f, DIRS["model"] / f)
atomic_write_json(DIRS["config"] / "config.json", {"cfg": asdict(cfg), "expected_splits": EXPECTED_SPLITS, "versions": VERSIONS, "label_mapping": LABEL2PUNCT,
                                                  "preprocessing_policy": "see punc_core.py docstring"})
tm = json.loads((DIRS["evaluation"] / "test_metrics.json").read_text()) if (DIRS["evaluation"] / "test_metrics.json").exists() else None
res_txt = (f"Test macro-F1 {tm['macro_f1']:.4f}, punctuation micro-F1 {tm['micro_f1_punct']:.4f}, exact-match {tm.get('exact_match_rate', float('nan')):.4f} "
           f"({'SUBSET: ' + str(len(test_sents)) + ' sentences' if tm['subset'] else 'full test split'}); measured by the training notebook.") if tm else "Not evaluated yet."
README = f"""# Persian punctuation restoration - ALBERT-fa (ZWNJ)

Token classification model fine-tuned from `{cfg.model_name}` on `{cfg.dataset_name}`.
Scope of training: **{meta['scope']}** ({summary.get('examples_read', 0):,} stream rows consumed, {summary.get('global_step')} optimizer steps).

## Install
`pip install torch transformers sentencepiece protobuf safetensors`

## Use
```python
import sys; sys.path.insert(0, "<this folder>")
from punc_inference import PunctuationRestorer
r = PunctuationRestorer("<this folder>")
r.restore("سلام حال شما چطور است")        # str -> str
r.restore(["...", "..."])                  # list -> list
```
Labels: {', '.join(f'{k}={v!r}' for k, v in LABEL2PUNCT.items())}.

## Behaviour and limitations
* Only punctuation is inserted; words are never changed (verified on every call, RuntimeError otherwise). Existing punctuation in the input is discarded and re-predicted.
* Supported marks: ، . ؟ ! ؛ :  Quotes, brackets, dashes and ellipses are not predicted; training examples with consecutive marks or ending-quote-after-punctuation patterns were excluded.
* Trained on written text (Wikipedia, blogs, news, Telegram...). Spoken/ASR transcripts may behave differently. Long inputs are windowed with overlap.
* Not evaluated against published scores; the numbers below are the only claims.

## Measured results
{res_txt}
Full metrics: `evaluation/test_metrics.json`, `evaluation/validation_metrics.json`, `evaluation/confusion_matrix_test.png`.

## Reproduce / evaluate
Run `PersianPunc_ALBERT_T4.ipynb` (sections 11-12 re-evaluate any checkpoint). Dataset access requires accepting the conditions on Hugging Face.
"""
(DIRS["model"] / "README.md").write_text(README)
sums = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(DIRS["model"].iterdir()) if p.is_file()}
(DIRS["model"] / "SHA256SUMS.json").write_text(json.dumps(sums, indent=1)); print("saved to", DIRS["model"], "->", list(sums))

In [ ]:
# ---- verification in a fresh process that only sees the Drive folder --------------------------------------------------
probe_in = EXAMPLES[:6]
script = ("import json,sys\nfrom punc_inference import PunctuationRestorer\n"
          f"r=PunctuationRestorer({str(DIRS['model'])!r})\nprint(json.dumps(r.restore(json.loads(sys.argv[1])),ensure_ascii=False))")
env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}; env["PYTHONPATH"] = str(DIRS["model"])
proc = subprocess.run([sys.executable, "-c", script, json.dumps(probe_in, ensure_ascii=False)], cwd=str(DIRS["model"]), env=env, capture_output=True, text=True)
assert proc.returncode == 0, proc.stderr[-2000:]
fresh_out = json.loads(proc.stdout.strip().splitlines()[-1]); assert fresh_out == batch[:6], "fresh-process output differs from in-process output"
sums = json.loads((DIRS["model"] / "SHA256SUMS.json").read_text())
for n, h in sums.items(): assert hashlib.sha256((DIRS["model"] / n).read_bytes()).hexdigest() == h, f"checksum mismatch {n}"
reloaded = AlbertForTokenClassification.from_pretrained(DIRS["model"]); assert reloaded.config.id2label == {i: l for i, l in ID2LABEL.items()} or \
    {int(k): v for k, v in reloaded.config.id2label.items()} == ID2LABEL
print("FINAL MODEL VERIFIED: loads from Google Drive in a fresh process, outputs identical, checksums match.")

## 15. Final report
Generated only from files and measurements produced above - nothing here is typed in by hand.

In [ ]:
def load_json(p):
    p = Path(p); return json.loads(p.read_text()) if p.exists() else None
S = load_json(DIRS["logs"] / "train_state_summary.json"); V = load_json(DIRS["evaluation"] / "validation_metrics.json")
T = load_json(DIRS["evaluation"] / "test_metrics.json"); PL = load_json(DIRS["logs"] / "pilot" / "pilot_result.json"); INTEG = load_json(DIRS["evaluation"] / "data_integrity.json")
report = {"hardware": GPU_INFO, "versions": VERSIONS, "fp16_autocast": USE_FP16, "saved_paths": {k: str(v) for k, v in DIRS.items()}}
if S:
    frac = S["examples_read"] / EXPECTED_SPLITS["train"]
    report["training"] = {"optimizer_steps": S["global_step"], "planned_steps": S["max_steps"], "completed_planned_steps": S["completed"], "stopped_early": S["stopped_early"],
        "stream_rows_consumed": S["examples_read"], "examples_trained_on": S["examples_used"], "fraction_of_train_split": frac,
        "scope": "FULL PASS" if frac >= 1 else "SUBSET-TRAINED", "sessions": S["sessions"], "train_hours": S["train_seconds"] / 3600, "best_val_macro_f1": S["best_metric"],
        "best_step": S["best_step"], "oom_skips": S["oom_skips"], "nonfinite_skips": S["nonfinite_skips"], "train_exclusions": S["exclusions"],
        "train_label_counts": dict(zip(LABELS, S["label_counts"]))}
if PL: report["pilot"] = PL
if INTEG: report["data_integrity"] = {k: INTEG[k] for k in ("exclusion_rate", "exclusions", "truncation_rate", "duplicate_rate_in_sample", "val_test_exact_overlap", "train_sample_in_val_or_test", "caveat")}
for k, R in (("validation", V), ("test", T)):
    if R:
        report[k] = {m: R.get(m) for m in ("macro_f1", "micro_f1_punct", "accuracy", "insertion_precision", "insertion_recall", "punctuation_error_rate",
                     "exact_match_rate", "lexical_change_rate", "exclusion_rate", "subset", "sentences_per_sec")}
        report[k]["baseline_no_punctuation_macro_f1"] = R["baseline_no_punctuation"]["macro_f1"]; report[k]["per_class"] = R["per_class"]
        report[k]["n_usable_sentences"] = R["counters"].get("n_valid")
if T and "latency" in T: report["inference_latency"] = T["latency"]
report["tests_executed_in_this_notebook"] = ["punc_core unit tests", "real-tokenizer alignment tests", "smoke test (forward/backward/masking/overfit/checkpoint reload/e2e)",
                                             "lexical-preservation tests", "fresh-process load from Drive", "checksum verification"]
report["known_limitations"] = [
    "Unless training_scope says FULL PASS, the model saw only a fraction of the training split; the published split sizes come from the dataset card.",
    "Sentences with consecutive marks, leading marks, or symbol-only tokens are excluded from training AND evaluation (rates reported); the model is untested on them.",
    "Only the six marks of the label inventory are predicted; quotes, brackets, dashes and ellipses are not.",
    "Data are written text from web/Wikipedia/Telegram sources filtered to end with proper punctuation; spoken-language (ASR) text may differ.",
    "Train/val/test leakage was checked by exact string match on a bounded train sample only.",
    "No published score is claimed; all numbers above were measured by this notebook."]
atomic_write_json(DIRS["evaluation"] / "final_report.json", report)
lines = ["# Final report", "", f"GPU: {GPU_INFO}", ""]
if S: lines += [f"* Training scope: **{report['training']['scope']}** - {S['examples_read']:,} stream rows ({frac:.3%} of train), {S['global_step']} steps, {S['train_seconds']/3600:.2f} h over {S['sessions']} session(s)"]
if V: lines += [f"* Validation macro-F1 {V['macro_f1']:.4f} (baseline {V['baseline_no_punctuation']['macro_f1']:.4f})"]
if T: lines += [f"* Test macro-F1 {T['macro_f1']:.4f}, exact match {T.get('exact_match_rate', float('nan')):.4f}, lexical change {T.get('lexical_change_rate')}, " + ("SUBSET evaluation" if T["subset"] else "full test split")]
lines += ["", "## Limitations"] + [f"* {x}" for x in report["known_limitations"]]
(DIRS["evaluation"] / "final_report.md").write_text("\n".join(lines)); print("\n".join(lines)); print("\n", json.dumps(report.get("training", {}), indent=1))